# 02 — Cleaning and leakage-safe splits

Run notebook 01 first. Configure `COLUMN_MAP` below using the actual dataset columns.
This notebook deliberately does not guess the schema or convert missing labels to negatives.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SPLITS_DIR = PROJECT_ROOT / "data" / "splits"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

# Update only after inspecting notebook 01:
# "file.csv": {"isoform": "CYP1A2", "smiles_col": "smiles",
#              "label_col": "substrate_label", "id_col": "compound_id"}
COLUMN_MAP = {}
print("Available CSV files:", [p.name for p in sorted(RAW_DIR.glob("*.csv"))])
print("COLUMN_MAP is currently empty; fill it after the audit.")

Available CSV files: ['cyp1a2_test.csv', 'cyp1a2_train.csv', 'cyp1a2_val.csv', 'cyp2c19_test.csv', 'cyp2c19_train.csv', 'cyp2c19_val.csv', 'cyp2c9_test.csv', 'cyp2c9_train.csv', 'cyp2c9_val.csv', 'cyp2d6_test.csv', 'cyp2d6_train.csv', 'cyp2d6_val.csv', 'cyp3a4_test.csv', 'cyp3a4_train.csv', 'cyp3a4_val.csv']
COLUMN_MAP is currently empty; fill it after the audit.


In [2]:
from rdkit import Chem

def canonicalize_smiles(value):
    if pd.isna(value):
        return None
    mol = Chem.MolFromSmiles(str(value))
    return Chem.MolToSmiles(mol, canonical=True) if mol is not None else None

def normalize_binary_label(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, np.integer, float, np.floating)) and value in (0, 1):
        return int(value)
    s = str(value).strip().lower()
    if s in {"1", "yes", "true", "substrate", "positive", "active"}:
        return 1
    if s in {"0", "no", "false", "non-substrate", "nonsubstrate", "negative", "inactive"}:
        return 0
    return np.nan

In [3]:
if not COLUMN_MAP:
    print("Configure COLUMN_MAP from the audit before running data cleaning.")
else:
    frames = []
    for filename, cfg in COLUMN_MAP.items():
        df = pd.read_csv(RAW_DIR / filename)
        for col in [cfg["smiles_col"], cfg["label_col"]]:
            if col not in df.columns:
                raise KeyError(f"{filename}: column {col!r} not found. Columns: {df.columns.tolist()}")
        part = pd.DataFrame({
            "source_file": filename,
            "cyp_isoform": cfg["isoform"],
            "smiles_original": df[cfg["smiles_col"]].astype("string"),
            "substrate_label_raw": df[cfg["label_col"]]
        })
        id_col = cfg.get("id_col")
        part["compound_id_source"] = df[id_col].astype("string") if id_col and id_col in df else pd.NA
        part["smiles_canonical"] = part["smiles_original"].map(canonicalize_smiles)
        part["substrate_label"] = part["substrate_label_raw"].map(normalize_binary_label).astype("Int64")
        frames.append(part)

    combined = pd.concat(frames, ignore_index=True)
    print("Combined shape:", combined.shape)
    print("Invalid/missing SMILES:", int(combined["smiles_canonical"].isna().sum()))
    print("Missing/unrecognized labels:", int(combined["substrate_label"].isna().sum()))
    display(combined.groupby("cyp_isoform")["substrate_label"].agg(["count", "mean"]))

Configure COLUMN_MAP from the audit before running data cleaning.


In [4]:
if "combined" in globals():
    cleaned = combined.dropna(subset=["smiles_canonical", "substrate_label"]).copy()
    cleaned["substrate_label"] = cleaned["substrate_label"].astype(int)
    conflicts = (cleaned.groupby(["smiles_canonical", "cyp_isoform"])["substrate_label"]
                 .nunique().reset_index(name="n_labels"))
    conflicts = conflicts[conflicts["n_labels"] > 1]
    print("Conflicting molecule/isoform pairs:", len(conflicts))
    display(conflicts.head(20))
    conflicts.to_csv(PROCESSED_DIR / "label_conflicts_to_review.csv", index=False)
    cleaned.to_csv(PROCESSED_DIR / "cyp_substrates_cleaned_pre_split.csv", index=False)
    print("Saved cleaned data and conflict review file.")
else:
    print("Run the configured cleaning cell first.")

Run the configured cleaning cell first.


In [5]:
!pip install scikit-learn


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:

from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "processed"
SPLITS_DIR = PROJECT_ROOT / "data" / "splits"
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

input_file = DATA_DIR / "cyp_substrates_cleaned.csv"
df = pd.read_csv(input_file)

# Every canonical compound receives exactly one global partition.
compounds = df["smiles_canonical"].dropna().unique()

train_compounds, remaining = train_test_split(
    compounds,
    test_size=0.30,
    random_state=42
)

val_compounds, test_compounds = train_test_split(
    remaining,
    test_size=0.50,
    random_state=42
)

split_map = {
    **{smiles: "train" for smiles in train_compounds},
    **{smiles: "val" for smiles in val_compounds},
    **{smiles: "test" for smiles in test_compounds}
}

df["global_split"] = df["smiles_canonical"].map(split_map)

# Verify that every compound belongs to one partition only.
assert df.groupby("smiles_canonical")["global_split"].nunique().max() == 1
assert df["global_split"].notna().all()

# Report compound-level split sizes.
split_sizes = pd.DataFrame({
    "partition": ["train", "val", "test"],
    "unique_compounds": [
        len(train_compounds),
        len(val_compounds),
        len(test_compounds)
    ]
})
split_sizes["percentage"] = (
    100 * split_sizes["unique_compounds"]
    / split_sizes["unique_compounds"].sum()
)

print("Global compound split sizes:")
display(split_sizes)

# Report record counts and class balance for every CYP task.
balance = (
    df.groupby(["isoform", "global_split"])
      .agg(
          records=("label", "size"),
          positives=("label", lambda s: (s == 1).sum()),
          negatives=("label", lambda s: (s == 0).sum()),
          unique_compounds=("smiles_canonical", "nunique")
      )
      .reset_index()
)

balance["positive_rate"] = balance["positives"] / balance["records"]

print("Per-isoform class balance:")
display(balance)

# Verify no canonical compound crosses global partitions.
global_overlap = []
sets = {
    p: set(df.loc[df["global_split"] == p, "smiles_canonical"])
    for p in ["train", "val", "test"]
}

for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    global_overlap.append({
        "partition_1": a,
        "partition_2": b,
        "shared_compounds": len(sets[a] & sets[b])
    })

overlap_df = pd.DataFrame(global_overlap)
print("Global split overlap — all values should be zero:")
display(overlap_df)

# Save the complete records with both original and global split assignments.
output_file = SPLITS_DIR / "cyp_substrates_global_cold_split.csv"
df.to_csv(output_file, index=False)

balance.to_csv(
    SPLITS_DIR / "global_cold_split_class_balance.csv",
    index=False
)
overlap_df.to_csv(
    SPLITS_DIR / "global_cold_split_overlap_check.csv",
    index=False
)

print("Saved global split:", output_file)


Global compound split sizes:


,partition,unique_compounds,percentage
0,train,1498,69.967305
1,val,321,14.992994
2,test,322,15.039701


Per-isoform class balance:


,isoform,global_split,records,positives,negatives,unique_compounds,positive_rate
0,CYP1A2,test,59,27,32,59,0.457627
1,CYP1A2,train,253,122,131,253,0.482213
2,CYP1A2,val,54,24,30,54,0.444444
3,CYP2C19,test,43,15,28,43,0.348837
4,CYP2C19,train,172,72,100,172,0.418605
5,CYP2C19,val,46,20,26,46,0.434783
6,CYP2C9,test,137,44,93,137,0.321168
7,CYP2C9,train,744,251,493,744,0.337366
8,CYP2C9,val,152,42,110,152,0.276316
9,CYP2D6,test,124,58,66,124,0.467742


Global split overlap — all values should be zero:


,partition_1,partition_2,shared_compounds
0,train,val,0
1,train,test,0
2,val,test,0


Saved global split: c:\Users\PAWAR ARAVIND\OneDrive\Desktop\CYP450_Substrate_Liability\data\splits\cyp_substrates_global_cold_split.csv


## Split policy

Create and save separate random/stratified, scaffold, and cold-compound splits.
Confirm that canonical compounds do not overlap across train and test. Use comparable
split definitions for single-task and multi-task models. Scaffold and cold-compound
splits must be documented precisely in the report.